# 5. Shortcuts: does a spurious cue take over the explanation?

In the paper's experiment (Section 5.2, Figure 5), a ResNet50 is trained for pneumonia on chest
X-rays where all negative images carry a small black box at a random place. When the box is present
at test time, one concept *saturates* a neighbourhood far larger than the box itself: direct
evidence that the encoder relies on the shortcut.

This notebook gives the measuring tool and runs it on a built-in encoder, which was *not* trained with
the shortcut, as a baseline. Run the same cells with your own model to test it.

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai @ git+https://github.com/Ahcene-B/NAVE.git"')
# elsewhere: pip install nave-xai
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cbook
from matplotlib.image import imread
from nave_xai import NAVEImage

image = imread(cbook.get_sample_data("grace_hopper.jpg", asfileobj=False))   # ships with matplotlib

def show(ax, explainer, exp, title=""):
    """An explanation map in the explainer's concept colours."""
    ax.imshow(explainer.palette[exp.explanation]); ax.set_title(title, fontsize=10); ax.axis("off")

TRAINING = "imagenet"     # "random" to skip downloads

def add_box(img, top, left, size):
    out = img.copy()
    out[top:top + size, left:left + size] = 0
    return out

box = (420, 330, 40)                      # top, left, size in pixels
marked = add_box(image, *box)

### The saturation measure

The share of the box's neighbourhood (three box sizes around it) covered by concepts that also cover
the box, with the box minus without it. Near 0: the box barely changes the explanation. Large: the
box's concept spreads, as with a learned shortcut. Fitting the concepts on the clean image and
explaining both versions with them keeps the concepts comparable.

In [ ]:
def saturation(clean, marked, box, ring=3):
    top, left, size = box
    t0, l0 = max(0, top - ring * size), max(0, left - ring * size)
    t1, l1 = top + (ring + 1) * size, left + (ring + 1) * size
    def covered(concepts):
        inside = np.unique(concepts[top:top + size, left:left + size])
        return np.isin(concepts[t0:t1, l0:l1], inside).mean()
    return covered(marked) - covered(clean)

results = {}
for model, layers in [("resnet50", "N34"), ("vitt16", "N:12")]:
    ex = NAVEImage(model=model, training=TRAINING, layers=layers, k=10).fit([image])
    clean, with_box = ex.predict([image, marked])
    results[model] = (ex, clean, with_box)
    print(f"{model}: saturation {saturation(clean.explanation, with_box.explanation, box):+.2f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8.5))
top, left, size = box
for row, (model, (ex, clean, with_box)) in enumerate(results.items()):
    axes[row, 0].imshow(marked); axes[row, 0].set_title("image with the box"); axes[row, 0].axis("off")
    show(axes[row, 1], ex, clean, f"{model}, clean")
    show(axes[row, 2], ex, with_box, f"{model}, with the box")
    for ax in axes[row]:
        ax.add_patch(plt.Rectangle((left, top), size, size, fill=False, ec="red", lw=1.5))
fig.tight_layout()

### Testing your own model

Fit the concepts on clean images, then compare explanations of the same images with and without the cue:

```python
explainer = NAVEImage(my_model, layers=["layer3", "layer4"], k=10).fit(clean_images)
clean = explainer.predict(test_images)
marked = explainer.predict([add_box(im, *box) for im in test_images])
scores = [saturation(c.explanation, m.explanation, box) for c, m in zip(clean, marked)]
```

A consistently large saturation means the cue dominates the encoder's representation near it.

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  editor    = {Ceci, Michelangelo and Kocev, Dragi and Monreale, Anna and Vens, Celine and Flach, Peter
               and Manco, Giuseppe and Spiliopoulou, Myra and Masciari, Elio},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26},
  isbn      = {978-3-032-37664-0}
}
```